In [0]:
SELECT *, _metadata.file_name AS file_name
FROM read_files('/Volumes/ecom_dev/bronze/landing/orders/ingest_date=2025-06-10/',
                format => 'csv', header => true)
LIMIT 10;

SELECT _metadata.file_name AS file_name, count(*) AS row_count
FROM read_files('/Volumes/ecom_dev/bronze/landing/orders/ingest_date=2025-06-10/',
                format => 'csv', header => true)
GROUP BY 1 ORDER BY 1;
SELECT * FROM read_files('/Volumes/ecom_dev/bronze/landing/orders/ingest_date=2026-02-28/',
                         format => 'csv', header => true) LIMIT 3;

SELECT * FROM read_files('/Volumes/ecom_dev/bronze/landing/orders/ingest_date=2026-03-01/',
                         format => 'csv', header => true) LIMIT 3;
SELECT _metadata.file_name AS file_name, count(*) AS row_count
FROM read_files('/Volumes/ecom_dev/bronze/landing/orders/ingest_date=2025-12-25/',
                format => 'csv', header => true)
GROUP BY 1;

WITH expected AS (
  SELECT explode(sequence(to_date('2024-11-01'), to_date('2026-11-01'), interval 1 month)) AS d
),
loaded AS (
  SELECT DISTINCT to_date(regexp_extract(file_path, 'snapshot_date=([0-9-]+)', 1)) AS d
  FROM ecom_dev.control.file_ingestion_log
  WHERE table_name = 'sellers'
)
SELECT e.d AS missing_snapshot
FROM expected e LEFT ANTI JOIN loaded l ON e.d = l.d;

LIST '/Volumes/ecom_dev/bronze/landing/sellers/';

MERGE INTO ecom_dev.control.pipeline_run_audit a
USING (
  SELECT run_id,
         concat('bronze.', table_name) AS table_name,
         sum(row_count) AS rows_read,
         sum(CASE WHEN status = 'LOADED' THEN row_count ELSE 0 END) AS rows_written,
         sum(CASE WHEN status = 'SKIPPED_DUPLICATE' THEN row_count ELSE 0 END) AS rows_rejected
  FROM ecom_dev.control.file_ingestion_log
  GROUP BY run_id, table_name
) s
ON a.layer = 'bronze'
   AND a.run_id = s.run_id
   AND a.table_name = s.table_name
   AND coalesce(a.rows_read, 0) = 0
WHEN MATCHED THEN UPDATE SET
  a.rows_read = s.rows_read,
  a.rows_written = s.rows_written,
  a.rows_rejected = s.rows_rejected;


SELECT table_name, status, rows_read, rows_written, rows_rejected, start_ts
FROM ecom_dev.control.pipeline_run_audit
WHERE layer = 'bronze'
ORDER BY start_ts, table_name;

CREATE TABLE IF NOT EXISTS ecom_dev.control.dq_rules (
  rule_id     STRING    NOT NULL,
  table_name  STRING    NOT NULL,
  rule_name   STRING    NOT NULL,
  rule_expr   STRING    NOT NULL,
  severity    STRING    NOT NULL,
  is_active   BOOLEAN   NOT NULL,
  updated_at  TIMESTAMP NOT NULL
) COMMENT 'Data quality rules, evaluated in Silver';

MERGE INTO ecom_dev.control.dq_rules AS t
USING (
  SELECT * FROM VALUES
    ('orders_01',      'orders',      'order_id_not_null',    'order_id IS NOT NULL',                                                              'quarantine'),
    ('orders_02',      'orders',      'customer_id_not_null', 'customer_id IS NOT NULL',                                                           'quarantine'),
    ('orders_03',      'orders',      'status_valid',         "order_status IN ('created','approved','shipped','delivered','canceled','returned')", 'quarantine'),
    ('orders_04',      'orders',      'purchase_ts_not_null', 'order_purchase_ts IS NOT NULL',                                                     'quarantine'),
    ('order_items_01', 'order_items', 'quantity_positive',    'quantity > 0',                                                                      'quarantine'),
    ('order_items_02', 'order_items', 'unit_price_valid',     'unit_price IS NOT NULL AND unit_price >= 0',                                        'quarantine'),
    ('payments_01',    'payments',    'payment_value_valid',  'payment_value IS NOT NULL AND payment_value >= 0',                                  'quarantine'),
    ('reviews_01',     'reviews',     'score_in_range',       'review_score BETWEEN 1 AND 5',                                                      'quarantine'),
    ('customers_01',   'customers',   'email_has_at',         "email LIKE '%@%'",                                                                  'warn'),
    ('products_01',    'products',    'price_positive',       'unit_price > 0',                                                                    'quarantine')
  AS v(rule_id, table_name, rule_name, rule_expr, severity)
) AS s
ON t.rule_id = s.rule_id
WHEN MATCHED THEN UPDATE SET
  t.table_name = s.table_name, t.rule_name = s.rule_name, t.rule_expr = s.rule_expr,
  t.severity = s.severity, t.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (rule_id, table_name, rule_name, rule_expr, severity, is_active, updated_at)
  VALUES (s.rule_id, s.table_name, s.rule_name, s.rule_expr, s.severity, true, current_timestamp());


SELECT rule_id, table_name, rule_name, severity, is_active FROM ecom_dev.control.dq_rules ORDER BY rule_id;

CREATE TABLE IF NOT EXISTS ecom_dev.control.dq_results (
  run_id        STRING    NOT NULL,
  table_name    STRING    NOT NULL,
  rule_id       STRING    NOT NULL,
  rule_name     STRING    NOT NULL,
  severity      STRING    NOT NULL,
  total_rows    BIGINT    NOT NULL,
  failed_rows   BIGINT    NOT NULL,
  pass_rate     DOUBLE    NOT NULL,
  evaluated_at  TIMESTAMP NOT NULL
) COMMENT 'Data quality results per run and rule';

CREATE TABLE IF NOT EXISTS ecom_dev.control.quarantine_records (
  run_id        STRING    NOT NULL,
  table_name    STRING    NOT NULL,
  rule_name     STRING    NOT NULL,
  record_key    STRING,
  record_json   STRING    NOT NULL,
  source_file   STRING,
  rejected_at   TIMESTAMP NOT NULL
) COMMENT 'Rows that failed a quarantine-severity rule, kept with the reason';

CREATE TABLE IF NOT EXISTS ecom_dev.silver.orders (
  order_id                 STRING    NOT NULL,
  customer_id              STRING,
  order_status             STRING,
  order_purchase_ts        TIMESTAMP,
  order_approved_ts        TIMESTAMP,
  order_shipped_ts         TIMESTAMP,
  order_delivered_ts       TIMESTAMP,
  estimated_delivery_date  DATE,
  order_date               DATE,
  channel                  STRING,
  updated_at               TIMESTAMP,
  _ingest_ts               TIMESTAMP,
  _source_file             STRING,
  _processed_ts            TIMESTAMP,
  _run_id                  STRING
) COMMENT 'One row per order, latest known state';

SELECT count(*) AS rows_in_silver, count(DISTINCT order_id) AS distinct_orders
FROM ecom_dev.silver.orders;
SELECT order_status, count(*) AS orders FROM ecom_dev.silver.orders GROUP BY 1 ORDER BY 2 DESC;
SELECT rule_name, severity, failed_rows, round(pass_rate, 4) AS pass_rate
FROM ecom_dev.control.dq_results WHERE table_name = 'orders' ORDER BY rule_id;
SELECT rule_name, count(*) AS rows FROM ecom_dev.control.quarantine_records
WHERE table_name = 'orders' GROUP BY 1;
SELECT table_name, last_watermark_value, last_success_ts
FROM ecom_dev.control.watermark WHERE table_name = 'silver.orders';
CREATE TABLE IF NOT EXISTS ecom_dev.silver.order_items (
  order_id         STRING        NOT NULL,
  order_item_id    INT           NOT NULL,
  product_id       STRING,
  seller_id        STRING,
  quantity         INT,
  unit_price       DECIMAL(12,2),
  discount_amount  DECIMAL(12,2),
  freight_value    DECIMAL(12,2),
  line_amount      DECIMAL(14,2),
  updated_at       TIMESTAMP,
  _ingest_ts       TIMESTAMP,
  _source_file     STRING,
  _processed_ts    TIMESTAMP,
  _run_id          STRING
) COMMENT 'One row per order line, latest known state';

MERGE INTO ecom_dev.control.dq_rules AS t
USING (
  SELECT * FROM VALUES
    ('order_items_03', 'order_items', 'order_id_not_null',      'order_id IS NOT NULL',      'quarantine'),
    ('order_items_04', 'order_items', 'order_item_id_not_null', 'order_item_id IS NOT NULL', 'quarantine')
  AS v(rule_id, table_name, rule_name, rule_expr, severity)
) AS s
ON t.rule_id = s.rule_id
WHEN MATCHED THEN UPDATE SET
  t.table_name = s.table_name, t.rule_name = s.rule_name, t.rule_expr = s.rule_expr,
  t.severity = s.severity, t.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (rule_id, table_name, rule_name, rule_expr, severity, is_active, updated_at)
  VALUES (s.rule_id, s.table_name, s.rule_name, s.rule_expr, s.severity, true, current_timestamp());


  CREATE TABLE IF NOT EXISTS ecom_dev.silver.payments (
  order_id              STRING        NOT NULL,
  payment_sequential    INT           NOT NULL,
  payment_type          STRING,
  payment_installments  INT,
  payment_value         DECIMAL(12,2),
  updated_at            TIMESTAMP,
  _ingest_ts            TIMESTAMP,
  _source_file          STRING,
  _processed_ts         TIMESTAMP,
  _run_id               STRING
) COMMENT 'One row per payment, latest known state';

CREATE TABLE IF NOT EXISTS ecom_dev.silver.reviews (
  review_id       STRING    NOT NULL,
  order_id        STRING,
  review_score    INT,
  review_comment  STRING,
  review_ts       TIMESTAMP,
  _ingest_ts      TIMESTAMP,
  _source_file    STRING,
  _processed_ts   TIMESTAMP,
  _run_id         STRING
) COMMENT 'One row per review';

CREATE TABLE IF NOT EXISTS ecom_dev.silver.customers (
  customer_id    STRING    NOT NULL,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  signup_date    DATE,
  updated_at     TIMESTAMP NOT NULL,
  _ingest_ts     TIMESTAMP,
  _source_file   STRING,
  _processed_ts  TIMESTAMP,
  _run_id        STRING
) COMMENT 'One row per customer VERSION (history kept for SCD2 in Gold)';

CREATE TABLE IF NOT EXISTS ecom_dev.silver.products (
  product_id     STRING        NOT NULL,
  product_name   STRING,
  category       STRING,
  brand          STRING,
  unit_price     DECIMAL(12,2),
  cost_price     DECIMAL(12,2),
  is_active      BOOLEAN,
  seller_id      STRING,
  updated_at     TIMESTAMP     NOT NULL,
  _ingest_ts     TIMESTAMP,
  _source_file   STRING,
  _processed_ts  TIMESTAMP,
  _run_id        STRING
) COMMENT 'One row per product VERSION (history kept for SCD2 in Gold)';

CREATE TABLE IF NOT EXISTS ecom_dev.silver.sellers (
  seller_id      STRING       NOT NULL,
  seller_name    STRING,
  city           STRING,
  state          STRING,
  seller_since   DATE,
  seller_rating  DECIMAL(3,1),
  updated_at     TIMESTAMP,
  _ingest_ts     TIMESTAMP,
  _source_file   STRING,
  _processed_ts  TIMESTAMP,
  _run_id        STRING
) COMMENT 'One row per seller, latest known state';

MERGE INTO ecom_dev.control.dq_rules AS t
USING (
  SELECT * FROM VALUES
    ('payments_02',  'payments',  'order_id_not_null',            'order_id IS NOT NULL',            'quarantine'),
    ('payments_03',  'payments',  'payment_sequential_not_null',  'payment_sequential IS NOT NULL',  'quarantine'),
    ('reviews_02',   'reviews',   'review_id_not_null',           'review_id IS NOT NULL',           'quarantine'),
    ('customers_02', 'customers', 'customer_id_not_null',         'customer_id IS NOT NULL',         'quarantine'),
    ('customers_03', 'customers', 'updated_at_not_null',          'updated_at IS NOT NULL',          'quarantine'),
    ('products_02',  'products',  'product_id_not_null',          'product_id IS NOT NULL',          'quarantine'),
    ('products_03',  'products',  'updated_at_not_null',          'updated_at IS NOT NULL',          'quarantine'),
    ('sellers_01',   'sellers',   'seller_id_not_null',           'seller_id IS NOT NULL',           'quarantine')
  AS v(rule_id, table_name, rule_name, rule_expr, severity)
) AS s
ON t.rule_id = s.rule_id
WHEN MATCHED THEN UPDATE SET
  t.table_name = s.table_name, t.rule_name = s.rule_name, t.rule_expr = s.rule_expr,
  t.severity = s.severity, t.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (rule_id, table_name, rule_name, rule_expr, severity, is_active, updated_at)
  VALUES (s.rule_id, s.table_name, s.rule_name, s.rule_expr, s.severity, true, current_timestamp());



  UPDATE ecom_dev.control.pipeline_config
SET primary_keys = array('customer_id', 'updated_at'), updated_at = current_timestamp()
WHERE source_table = 'customers';

UPDATE ecom_dev.control.pipeline_config
SET primary_keys = array('product_id', 'updated_at'), updated_at = current_timestamp()
WHERE source_table = 'products';

SELECT table_name, count(*) AS columns
FROM ecom_dev.information_schema.columns
WHERE table_schema = 'silver'
GROUP BY 1 ORDER BY 1;

SELECT table_name, count(*) AS rules FROM ecom_dev.control.dq_rules GROUP BY 1 ORDER BY 1;

SELECT 'orders' AS tbl, count(*) AS silver_rows, count(DISTINCT order_id) AS keys FROM ecom_dev.silver.orders
UNION ALL SELECT 'order_items', count(*), count(DISTINCT order_id, order_item_id) FROM ecom_dev.silver.order_items
UNION ALL SELECT 'payments',    count(*), count(DISTINCT order_id, payment_sequential) FROM ecom_dev.silver.payments
UNION ALL SELECT 'reviews',     count(*), count(DISTINCT review_id) FROM ecom_dev.silver.reviews
UNION ALL SELECT 'customers',   count(*), count(DISTINCT customer_id, updated_at) FROM ecom_dev.silver.customers
UNION ALL SELECT 'products',    count(*), count(DISTINCT product_id, updated_at) FROM ecom_dev.silver.products
UNION ALL SELECT 'sellers',     count(*), count(DISTINCT seller_id) FROM ecom_dev.silver.sellers;

SELECT table_name, rule_name, count(*) AS rows
FROM ecom_dev.control.quarantine_records
GROUP BY 1, 2 ORDER BY 1, 2;